In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
from numpy import loadtxt
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout, BatchNormalization, Activation
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler
from numpy import exp
from tensorflow.keras import Model

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import accuracy_score,mean_absolute_error,mean_squared_error
from sklearn.model_selection import GridSearchCV
from keras.wrappers.scikit_learn import KerasRegressor,KerasClassifier


pd.set_option('max_rows',999)

ModuleNotFoundError: No module named 'torch'

### Define Function

In [2]:
def exp_decay(self, epoch):
    initial_lrate = 0.001
    k = 0.1
    lrate = initial_lrate * exp(-k*epoch)
    return lrate


def create_model_neurons(optimizer='Adam',layers=1,dropout = 0.15, neurons=64,epochs=100,norm=True, activation='relu'):
    # create model
    model = Sequential()
    model.add(Dense(neurons, input_dim=22, kernel_initializer='uniform'))
    if norm:
        model.add(BatchNormalization())
    model.add(Activation(activation))
    model.add(Dropout(dropout))
    for layer in range(layers):
        model.add(Dense(neurons, kernel_initializer='uniform'))
        if norm:
            model.add(BatchNormalization())
        model.add(Activation(activation))
        model.add(Dropout(dropout))
    model.add(Dense(1))
    # Compile model
    model.compile(loss='mean_squared_error',optimizer=optimizer, metrics=['mse'])
    return model

def build_model():
    model = Sequential()
    model.add(Dense(64, activation='relu', input_shape=(13,)))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1))

    model.compile(optimizer='rmsprop',
              loss='mse',
              metrics=['mae'])
    return model

### Import data

In [3]:
df_sunspots = pd.read_csv('data/data.csv.txt')

In [4]:
# Split month
df_sunspots.columns = ['Data','Sunspots']
df_sunspots['Month'] = df_sunspots.Data.apply(lambda x: int(x.split('-')[1]))

#build phase space
m = 10
df_phase=pd.DataFrame([df_sunspots.Sunspots.iloc[-m+d:d].values for d in range(df_sunspots.shape[0])])
df_phase.columns =['Sunspots_'+str(col-10) for col in df_phase.columns]
df_target = df_sunspots['Sunspots'].iloc[m:]

#get month dummies 
df_data=pd.concat([pd.get_dummies(df_sunspots.Month,prefix='Month_'),df_phase],axis=1).iloc[m:]
df_data.set_index(df_sunspots['Data'].iloc[m:],inplace=True)

# MPL

In [95]:
X_train, X_test, y_train, y_test = train_test_split(df_data.values, df_target.values, test_size=0.3, random_state=42)

In [72]:
# create model

model = KerasRegressor(build_fn=create_model_neurons, epochs=100, batch_size=10, verbose=1)

lr_schedule_1 = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.1,
    decay_steps=100000,
    decay_rate=0.01)
lr_schedule_2 = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.5,
    decay_steps=100000,
    decay_rate=0.1)
lr_schedule_3 = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.05,
    decay_steps=100000,
    decay_rate=0.01)

#define the grid search parameters
optimizer = [tf.keras.optimizers.Adam(learning_rate=lr_schedule_1),
             #tf.keras.optimizers.Adam(learning_rate=lr_schedule_2),
             tf.keras.optimizers.Adam(learning_rate=lr_schedule_3),]

epochs=[400]
neurons = [60,80]
layers =[0,1,2]
dropout=[ .25]
norm=[True]
activation = ['relu']
param_grid = dict(epochs=epochs,layers=layers,dropout=dropout,optimizer=optimizer,neurons=neurons, norm=norm, activation=activation)

callabacks_list = [EarlyStopping(patience=50,monitor='loss')]
# define the grid search parameters
tscv=TimeSeriesSplit(3, test_size=10)

grid_mlp = GridSearchCV(estimator=model, param_grid=param_grid, cv=tscv,verbose=1)
c = grid_mlp.fit(X_train, y_train, batch_size=16,callbacks=callabacks_list)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Epoch 1/400


C:\Users\diego\AppData\Local\Temp/ipykernel_387308/2345438006.py:3: DeprecationWarning: KerasRegressor is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  model = KerasRegressor(build_fn=create_model_neurons, epochs=100, batch_size=10, verbose=1)


139/139 [==============================] - 1s 2ms/step - loss: 659.3230 - mse: 659.3230
Epoch 2/400
139/139 [==============================] - 0s 2ms/step - loss: 480.4230 - mse: 480.4230
Epoch 3/400
139/139 [==============================] - 0s 1ms/step - loss: 490.4348 - mse: 490.4348
Epoch 4/400
139/139 [==============================] - 0s 2ms/step - loss: 539.1410 - mse: 539.1410
Epoch 5/400
139/139 [==============================] - 0s 1ms/step - loss: 471.8256 - mse: 471.8256
Epoch 6/400
139/139 [==============================] - 0s 2ms/step - loss: 556.3113 - mse: 556.3113
Epoch 7/400
139/139 [==============================] - 0s 2ms/step - loss: 472.2158 - mse: 472.2158
Epoch 8/400
139/139 [==============================] - 0s 1ms/step - loss: 500.7066 - mse: 500.7066
Epoch 9/400
139/139 [==============================] - 0s 2ms/step - loss: 486.0410 - mse: 486.0410
Epoch 10/400
139/139 [==============================] - 0s 2ms/step - loss: 477.6093 - mse: 477.6093
Epoch 11/40

140/140 [==============================] - 1s 2ms/step - loss: 673.6174 - mse: 673.6174
Epoch 2/400
140/140 [==============================] - 0s 2ms/step - loss: 545.2267 - mse: 545.2267
Epoch 3/400
140/140 [==============================] - 0s 2ms/step - loss: 483.0872 - mse: 483.0872
Epoch 4/400
140/140 [==============================] - 0s 2ms/step - loss: 516.7640 - mse: 516.7640
Epoch 5/400
140/140 [==============================] - 0s 2ms/step - loss: 543.4711 - mse: 543.4711
Epoch 6/400
140/140 [==============================] - 0s 2ms/step - loss: 502.7011 - mse: 502.7011
Epoch 7/400
140/140 [==============================] - 0s 2ms/step - loss: 494.3102 - mse: 494.3102
Epoch 8/400
140/140 [==============================] - 0s 2ms/step - loss: 514.2746 - mse: 514.2746
Epoch 9/400
140/140 [==============================] - 0s 1ms/step - loss: 503.2996 - mse: 503.2996
Epoch 10/400
140/140 [==============================] - 0s 1ms/step - loss: 537.4288 - mse: 537.4288
Epoch 11/40

140/140 [==============================] - 0s 2ms/step - loss: 413.0495 - mse: 413.0495
Epoch 163/400
140/140 [==============================] - 0s 2ms/step - loss: 413.2278 - mse: 413.2278
Epoch 164/400
140/140 [==============================] - 0s 1ms/step - loss: 415.6011 - mse: 415.6011
Epoch 165/400
140/140 [==============================] - 0s 1ms/step - loss: 416.3909 - mse: 416.3909
Epoch 166/400
140/140 [==============================] - 0s 1ms/step - loss: 448.3699 - mse: 448.3699
Epoch 167/400
140/140 [==============================] - 0s 2ms/step - loss: 462.4504 - mse: 462.4504
Epoch 168/400
140/140 [==============================] - 0s 2ms/step - loss: 419.9612 - mse: 419.9612
Epoch 169/400
140/140 [==============================] - 0s 1ms/step - loss: 444.4190 - mse: 444.4190
Epoch 170/400
140/140 [==============================] - 0s 1ms/step - loss: 412.8610 - mse: 412.8610
Epoch 171/400
140/140 [==============================] - 0s 1ms/step - loss: 408.2957 - mse: 408

139/139 [==============================] - 0s 2ms/step - loss: 448.8141 - mse: 448.8141
Epoch 13/400
139/139 [==============================] - 0s 2ms/step - loss: 512.0699 - mse: 512.0699
Epoch 14/400
139/139 [==============================] - 0s 1ms/step - loss: 466.8929 - mse: 466.8929
Epoch 15/400
139/139 [==============================] - 0s 1ms/step - loss: 467.2770 - mse: 467.2770
Epoch 16/400
139/139 [==============================] - 0s 1ms/step - loss: 452.7675 - mse: 452.7675
Epoch 17/400
139/139 [==============================] - 0s 2ms/step - loss: 454.7984 - mse: 454.7984
Epoch 18/400
139/139 [==============================] - 0s 2ms/step - loss: 447.5165 - mse: 447.5165
Epoch 19/400
139/139 [==============================] - 0s 1ms/step - loss: 469.8986 - mse: 469.8986
Epoch 20/400
139/139 [==============================] - 0s 1ms/step - loss: 475.7451 - mse: 475.7451
Epoch 21/400
139/139 [==============================] - 0s 2ms/step - loss: 438.2382 - mse: 438.2382
Epo

... (33 outputs left out)

139/139 [==============================] - 0s 2ms/step - loss: 465.3831 - mse: 465.3831
Epoch 36/400
139/139 [==============================] - 0s 2ms/step - loss: 431.8788 - mse: 431.8788
Epoch 37/400
139/139 [==============================] - 0s 2ms/step - loss: 441.7303 - mse: 441.7303
Epoch 38/400
139/139 [==============================] - 0s 2ms/step - loss: 441.8327 - mse: 441.8327
Epoch 39/400
139/139 [==============================] - 0s 2ms/step - loss: 448.2998 - mse: 448.2998
Epoch 40/400
139/139 [==============================] - 0s 3ms/step - loss: 490.2359 - mse: 490.2359
Epoch 41/400
139/139 [==============================] - 0s 2ms/step - loss: 461.0921 - mse: 461.0921
Epoch 42/400
139/139 [==============================] - 0s 3ms/step - loss: 460.2005 - mse: 460.2005
Epoch 43/400
139/139 [==============================] - 0s 3ms/step - loss: 476.5494 - mse: 476.5494
Epoch 44/400
139/139 [==============================] - 0s 2ms/step - loss: 414.4254 - mse: 414.4254
Epo

140/140 [==============================] - 0s 3ms/step - loss: 439.0799 - mse: 439.0799
Epoch 84/400
140/140 [==============================] - 0s 3ms/step - loss: 437.7638 - mse: 437.7638
Epoch 85/400
140/140 [==============================] - 0s 2ms/step - loss: 477.9188 - mse: 477.9188
Epoch 86/400
140/140 [==============================] - 0s 2ms/step - loss: 458.5678 - mse: 458.5678
Epoch 87/400
140/140 [==============================] - 0s 2ms/step - loss: 442.7664 - mse: 442.7664
Epoch 88/400
140/140 [==============================] - 0s 2ms/step - loss: 476.0048 - mse: 476.0048
Epoch 89/400
140/140 [==============================] - 0s 2ms/step - loss: 433.8958 - mse: 433.8958
Epoch 90/400
140/140 [==============================] - 0s 2ms/step - loss: 461.2528 - mse: 461.2528
Epoch 91/400
140/140 [==============================] - 0s 2ms/step - loss: 475.7853 - mse: 475.7853
Epoch 92/400
140/140 [==============================] - 0s 2ms/step - loss: 465.7577 - mse: 465.7577
Epo

140/140 [==============================] - 0s 2ms/step - loss: 501.5967 - mse: 501.5967
Epoch 17/400
140/140 [==============================] - 0s 2ms/step - loss: 458.2348 - mse: 458.2348
Epoch 18/400
140/140 [==============================] - 0s 2ms/step - loss: 463.0215 - mse: 463.0215
Epoch 19/400
140/140 [==============================] - 0s 2ms/step - loss: 479.6327 - mse: 479.6327
Epoch 20/400
140/140 [==============================] - 0s 2ms/step - loss: 453.7204 - mse: 453.7204
Epoch 21/400
140/140 [==============================] - 0s 2ms/step - loss: 455.8038 - mse: 455.8038
Epoch 22/400
140/140 [==============================] - 0s 2ms/step - loss: 484.8871 - mse: 484.8871
Epoch 23/400
140/140 [==============================] - 0s 2ms/step - loss: 477.3709 - mse: 477.3709
Epoch 24/400
140/140 [==============================] - 0s 2ms/step - loss: 463.0544 - mse: 463.0544
Epoch 25/400
140/140 [==============================] - 0s 2ms/step - loss: 499.2086 - mse: 499.2086
Epo

141/141 [==============================] - 0s 1ms/step - loss: 463.4483 - mse: 463.4483
Epoch 15/400
141/141 [==============================] - 0s 1ms/step - loss: 462.9056 - mse: 462.9056
Epoch 16/400
141/141 [==============================] - 0s 2ms/step - loss: 460.0254 - mse: 460.0254
Epoch 17/400
141/141 [==============================] - 0s 1ms/step - loss: 464.4328 - mse: 464.4328
Epoch 18/400
141/141 [==============================] - 0s 1ms/step - loss: 481.7794 - mse: 481.7794
Epoch 19/400
141/141 [==============================] - 0s 1ms/step - loss: 485.9360 - mse: 485.9360
Epoch 20/400
141/141 [==============================] - 0s 1ms/step - loss: 475.9974 - mse: 475.9974
Epoch 21/400
141/141 [==============================] - 0s 2ms/step - loss: 499.8505 - mse: 499.8505
Epoch 22/400
141/141 [==============================] - 0s 2ms/step - loss: 476.1357 - mse: 476.1357
Epoch 23/400
141/141 [==============================] - 0s 1ms/step - loss: 511.1039 - mse: 511.1039
Epo

In [73]:
# summarize results
print("Best: %f using %s" % (grid_mlp.best_score_, grid_mlp.best_params_))
means = grid_mlp.cv_results_['mean_test_score']
stds = grid_mlp.cv_results_['std_test_score']
params = grid_mlp.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: -238.416061 using {'activation': 'relu', 'dropout': 0.25, 'epochs': 400, 'layers': 0, 'neurons': 80, 'norm': True, 'optimizer': <keras.optimizer_v2.adam.Adam object at 0x0000025C6F933CD0>}
-248.802999 (79.293711) with: {'activation': 'relu', 'dropout': 0.25, 'epochs': 400, 'layers': 0, 'neurons': 60, 'norm': True, 'optimizer': <keras.optimizer_v2.adam.Adam object at 0x0000025C6F933CD0>}
-339.736043 (116.766634) with: {'activation': 'relu', 'dropout': 0.25, 'epochs': 400, 'layers': 0, 'neurons': 60, 'norm': True, 'optimizer': <keras.optimizer_v2.adam.Adam object at 0x0000025C6F933A30>}
-238.416061 (41.856779) with: {'activation': 'relu', 'dropout': 0.25, 'epochs': 400, 'layers': 0, 'neurons': 80, 'norm': True, 'optimizer': <keras.optimizer_v2.adam.Adam object at 0x0000025C6F933CD0>}
-323.460307 (211.101397) with: {'activation': 'relu', 'dropout': 0.25, 'epochs': 400, 'layers': 0, 'neurons': 80, 'norm': True, 'optimizer': <keras.optimizer_v2.adam.Adam object at 0x0000025C6F933A30>}

In [74]:
mlp_regressor=create_model_neurons()
mlp_regressor.fit(X_train, y_train, epochs=400, batch_size=16, verbose=0)

In [76]:
print(mean_absolute_error(y_test,mlp_regressor.predict(X_test)))
print(mean_squared_error(y_test,mlp_regressor.predict(X_test)))

12.598152318077155
313.22975592974916


In [86]:
model_mlp = KerasRegressor(build_fn=create_model_neurons, epochs=400, batch_size=16, verbose=1)
model_mlp.set_params(**grid_mlp.best_params_)
model_mlp.fit(X_train, y_train, epochs=400, batch_size=16, verbose=0)

C:\Users\diego\AppData\Local\Temp/ipykernel_387308/45125054.py:1: DeprecationWarning: KerasRegressor is deprecated, use Sci-Keras (https://github.com/adriangb/scikeras) instead. See https://www.adriangb.com/scikeras/stable/migration.html for help migrating.
  model_mlp = KerasRegressor(build_fn=create_model_neurons, epochs=400, batch_size=16, verbose=1)


In [131]:
grid_mlp

GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=3, test_size=10),
             estimator=<keras.wrappers.scikit_learn.KerasRegressor object at 0x0000025C6F933C70>,
             param_grid={'activation': ['relu'], 'dropout': [0.25],
                         'epochs': [400], 'layers': [0, 1, 2],
                         'neurons': [60, 80], 'norm': [True],
                         'optimizer': [<keras.optimizer_v2.adam.Adam object at 0x0000025C6F933CD0>,
                                       <keras.optimizer_v2.adam.Adam object at 0x0000025C6F933A30>]},
             verbose=1)

In [80]:
print(mean_absolute_error(y_test,grid_mlp.predict(X_test)))
print(mean_squared_error(y_test,grid_mlp.predict(X_test)))

57/57 [==============================] - 0s 1ms/step
12.081910098744457
57/57 [==============================] - 0s 1ms/step
266.1990290191032


In [87]:
print(mean_absolute_error(y_test,model_mlp.predict(X_test)))
print(mean_squared_error(y_test,model_mlp.predict(X_test)))

36/36 [==============================] - 0s 1ms/step
11.891084681880855
36/36 [==============================] - 0s 1ms/step
268.23229364015134


In [84]:
print(mean_absolute_error(y_test,model_mlp.predict(X_test)))
print(mean_squared_error(y_test,model_mlp.predict(X_test)))

36/36 [==============================] - 0s 1ms/step
12.035300434187214
36/36 [==============================] - 0s 857us/step
283.55933154716536


In [81]:
print(mean_absolute_error(y_test,model_mlp.predict(X_test)))
print(mean_squared_error(y_test,model_mlp.predict(X_test)))

57/57 [==============================] - 0s 1ms/step
12.104998133784935
57/57 [==============================] - 0s 1ms/step
266.5255741946008


# RandomForestRegressor

In [9]:
from sklearn.ensemble import RandomForestRegressor

In [60]:
param_grid = {'bootstrap': [True, False],
 'max_depth': [10, 20, 30],
 'max_features': ['auto', 'sqrt'],
 'min_samples_leaf': [1, 2, 4],
 'min_samples_split': [2, 5, 10],
 'n_estimators': [100,200, 300]}

In [61]:
# Create a based model
rf = RandomForestRegressor()

grid_search_rf = GridSearchCV(estimator = rf, param_grid = param_grid, 
                          cv = tscv, n_jobs = -1, verbose = 1)
grid_search_rf.fit(X_train, y_train)

Fitting 3 folds for each of 324 candidates, totalling 972 fits


GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=3, test_size=10),
             estimator=RandomForestRegressor(), n_jobs=-1,
             param_grid={'bootstrap': [True, False], 'max_depth': [10, 20, 30],
                         'max_features': ['auto', 'sqrt'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             verbose=2)

In [62]:
print(mean_absolute_error(y_test,grid_search_rf.predict(X_test)))
print(mean_squared_error(y_test,grid_search_rf.predict(X_test)))

12.11221881299953
289.8473747530901


In [64]:
rf_best = RandomForestRegressor()
rf_best.set_params(**grid_search_rf.best_params_)
rf_best.fit(X_train, y_train)

RandomForestRegressor(max_depth=20, min_samples_leaf=2)

In [65]:
print(mean_absolute_error(y_test,rf_best.predict(X_test)))
print(mean_squared_error(y_test,rf_best.predict(X_test)))

12.092063318235514
288.6466295447292


# XGBRegressor

In [88]:
from xgboost import XGBRegressor

In [170]:
tscv=TimeSeriesSplit(2, test_size=100)

In [193]:
X_train, X_test, y_train, y_test = train_test_split(df_data.values, df_target.values, test_size=0.7, random_state=42)

In [211]:
train_1=pd.DataFrame(X_train)
train_1['y']=y_train

train_2=pd.DataFrame(X_train)
train_2['y']=y_train
train_2=train_2.sort_values('y').reset_index(drop=True)

In [220]:
grid_search_XGB = GridSearchCV(
            estimator=XGBRegressor(),
            param_grid={
                        "max_depth": [ 8],

                        "colsample_bytree":[ 0.3, 0.4],},
            cv=tscv, scoring='neg_mean_squared_error', verbose=1)
grid_search_XGB.fit(train_1.drop(['y'],axis=1),train_1['y'],verbose=1)

Fitting 2 folds for each of 2 candidates, totalling 4 fits


GridSearchCV(cv=TimeSeriesSplit(gap=0, max_train_size=None, n_splits=2, test_size=100),
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=False, eval_metric=None,
                                    gamma=None, gpu_id=None, grow_policy=None,
                                    importance_type=None,
                                    in...
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None, missing=nan,
                                    monotone_constraints=None, n_estimators=100,
                         

In [221]:
print(mean_absolute_error(y_test,grid_search_XGB.predict(X_test)))
print(mean_squared_error(y_test,grid_search_XGB.predict(X_test)))

13.275825868717412
351.44533864890917


In [226]:
print(tf.test.is_gpu_available())
print(tf.test.is_built_with_cuda())

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
False
True


In [222]:
XGB_best = XGBRegressor()
XGB_best =XGB_best.set_params(**grid_search_XGB.best_params_)
XGB_best.fit(train_2.drop(['y'],axis=1),train_2['y'])

XGBRegressor(base_score=0.5, booster='gbtree', callbacks=None,
             colsample_bylevel=1, colsample_bynode=1, colsample_bytree=0.4,
             early_stopping_rounds=None, enable_categorical=False,
             eval_metric=None, gamma=0, gpu_id=-1, grow_policy='depthwise',
             importance_type=None, interaction_constraints='',
             learning_rate=0.300000012, max_bin=256, max_cat_to_onehot=4,
             max_delta_step=0, max_depth=8, max_leaves=0, min_child_weight=1,
             missing=nan, monotone_constraints='()', n_estimators=100, n_jobs=0,
             num_parallel_tree=1, predictor='auto', random_state=0, reg_alpha=0,
             reg_lambda=1, ...)

In [223]:
mean_absolute_error(y_test,XGB_best.predict(X_test)) == mean_absolute_error(y_test,grid_search_XGB.predict(X_test))

True

In [224]:
print(mean_absolute_error(y_test,XGB_best.predict(X_test)))
print(mean_squared_error(y_test,XGB_best.predict(X_test)))

13.275825868717412
351.44533864890917


# Results

In [23]:
print(mean_absolute_error(y_test,grid_search_mlp.predict(X_test)))
print(mean_squared_error(y_test,grid_search_mlp.predict(X_test)))

57/57 [==============================] - 0s 872us/step
11.86612542362824
57/57 [==============================] - 0s 1ms/step
272.2589589939081


In [12]:
print(mean_absolute_error(y_test,mlp_regressor.predict(X_test)))
print(mean_squared_error(y_test,mlp_regressor.predict(X_test)))

12.200511724244656
282.9980029818669


In [13]:
print(mean_absolute_error(y_test,grid_search_rf.predict(X_test)))
print(mean_squared_error(y_test,grid_search_rf.predict(X_test)))

12.41679347405833
300.1069709773603


In [15]:
print(mean_absolute_error(y_test,grid_search_XGB.predict(X_test)))
print(mean_squared_error(y_test,grid_search_XGB.predict(X_test)))

12.40975766606178
295.1027604646743
